In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from typing import List, Dict
import os
import sys
from urllib.parse import urljoin
from pprint import pprint
import re
from typing import Dict

BASE_PATH = "../../../data/RAG"

PROBABILITY_BASE_URL = "https://maplestory.nexon.com/Guide/CashShop/Probability/"

## 1. 엔드포인트 URL 확인
- 사이트 URL : `https://maplestory.nexon.com/Guide/CashShop/Probability`
- headers : `{"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36"}`

### ⚠️ 확률형 아이템의 세부 내용들의 URL이 규칙이 없음
- `https://maplestory.nexon.com/Guide/CashShop/Probability/RoyalStyle`
- `https://maplestory.nexon.com/Guide/CashShop/Probability/MasterpieceRed`
- `https://maplestory.nexon.com/Guide/CashShop/Probability/MasterpieceBlack`
따라서 다른 방법을 찾아야 함

### 💡 html 구조에서 해결방법을 찾음
```html
<div class="right_aside_new">
    <div class="percent_item_rnb">
        <ul>
            <li>
                <a href="https://maplestory.nexon.com/Guide/CashShop/Probability/RoyalStyle">치장성</a>
```
- html 구조에서 url을 찾음! -> 이 방법으로 수집 진행

In [ ]:
def get_probability_page(url: str) -> str:
    headers = {
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) " "Chrome/120.0.0.0")
    }

    try:
        response = requests.get(url, headers=headers, timeout=20)

        response.raise_for_status()

        return response.text

    except requests.exceptions.RequestException as e:
        print(f"페이지 요청 중 오류 발생: {e}")
        return ""

In [ ]:
def get_probability_urls():
    response = get_probability_page(PROBABILITY_BASE_URL)
    # response.text : 서버가 보낸 원문 문자열
    soup = BeautifulSoup(response, "html.parser")

    # 확률형 아이템을 리스트로 선언
    probability_items = []
    # BeautifulSoup으로 선언한 soup 변수로 위에서 확인한 html 구조(".percent_item_rnb a[href]") 가져오기 - a[href]에 문서 고유 url이 있기 때문
    links = soup.select(".percent_item_rnb a[href]")
    # link에 담겨있는 것 : [<a href="/Guide/CashShop/Probability/RoyalStyle">치장성</a>, <a href="/Guide/CashShop/Probability/RoyalStyle">로얄스타일</a>, ...]

    # links에 저장된 url 하나씩 꺼내기
    for link in links:
        # link를 text로 가져오고 link["href"]로 url 가져오기
        name = link.get_text(strip=True)
        href = link["href"]

        # BASE URL과 가져온 url을 붙임(기준 주소의 마지막 경로 구조에 맞춰 자연스럽게 이어짐)
        full_url = urljoin(PROBABILITY_BASE_URL, href)
        # 아까 선언한 리스트에 제목과 url 저장
        probability_items.append({"name": name, "url": full_url})

    return probability_items

#### `normalize_html_table`은 지피티로 바이브코딩 진행함
HTML 표의 `rowspan`, `colspan`을 실제 셀로 펼쳐서 모든 행이 같은 열 개수를 가지는 2차원 리스트로 만드는 역할

```html
<table>
    <tr>
        <th rowspan="2">아이템</th>
        <th colspan="2">확률</th>
    </tr>

    <tr>
        <th>등급</th>
        <th>확률</th>
    </tr>
</table>
```
HTML 자체는 행별 셀 수가 다르지만 함수를 정규화하면
```python
    ["아이템", "확률", "확률"],
    ["아이템", "등급", "확률"],
```
처럼 만들어줌

In [ ]:
def normalize_html_table(table):
    """
    HTML table의 rowspan, colspan을 실제 셀 값으로 펼쳐서
    모든 행의 열 개수가 동일한 2차원 리스트로 변환하는 함수

    예:
        <td rowspan="2">A</td>

    HTML에서는 A가 첫 번째 행에만 존재하지만,
    결과 데이터에서는 아래 행에도 A를 복사해서 넣어준다.
    """
    # 최종적으로 모든 행을 저장할 리스트
    rows = []

    # rowspan 때문에 미래의 행에 들어가야 할 값을 저장하는 딕셔너리
    # 미래 행에 채워져야 하는 rowspan 값 저장
    # key  : (행 번호, 열 번호)
    # value: 셀 내용
    spans = {}

    # 현재 table에 직접 포함된 tr만 수집하기 위한 리스트
    trs = []
    # recursive=False를 사용하는 이유 
    # : table 안에 또 다른 table이 있을 경우 중첩 table의 tr까지 가져오는 것을 방지하기 위함
    # <thread>, <tbody>, <tfoot> 안에 직접 존재하는 tr 수집
    for section in table.find_all(["thead", "tbody", "tfoot"], recursive=False):
        trs.extend(section.find_all("tr", recursive=False))

    # 일부 HTML은 tbody 없이 바로 tr이 있는 경우도 있음
    trs.extend(table.find_all("tr", recursive=False))

    # 표의 각 행(tr)을 순서대로 처리
    for row_index, tr in enumerate(trs):
        # 현재 행의 데이터를 저장할 리스트
        row = []
        # 현재 몇 번째 열을 처리하고 있는지 추적
        col_index = 0

        # 이전 행의 rowspan 때문에 현재 위치에 이미 들어가야 할 값이 있는 경우 그 값을 현재 행에 채워주는 내부 함수
        def fill_rowspan():
            nonlocal col_index

            # 현재 행/열 좌표가 spans에 등록되어 있다면 이전 rowsapn 값을 현재 행에 추가
            while (row_index, col_index) in spans:
                row.append(spans[(row_index, col_index)])
                # 다음 열로 이동
                col_index += 1

        # 행 시작 부분에 rowspan 값이 있을 수도 있으므로 실제 cell을 읽기 전에 먼저 처리
        fill_rowspan()

        # 현재 tr에 포함된 th, td 셀만 가져오기
        cells = tr.find_all(["th", "td"], recursive=False)

        # 현재 행의 각 셀 처리
        for cell in cells:
            # 새로운 셀을 넣기 전에 rowspan으로 예약되어 있으면 먼저 채움
            fill_rowspan()

            # 셀 내부의 텍스트 추출
            value = cell.get_text(" ", strip=True)
            # rowspan 값 가져오기(속성이 없으면 기본값 1)
            rowspan = int(cell.get("rowspan", 1) or 1)
            # colspan 값 가져오기(속성이 없으면 기본값 1)
            colspan = int(cell.get("colspan", 1) or 1)

            # colspan만큼 2라면 현재 value를 두 개의 열에 동일하게 넣음
            for offset in range(colspan):
                # 실제로 값이 들어갈 열 번호
                current_col = col_index + offset
                # 현재 행에 셀 값 추가
                row.append(value)

                # rowspan이 2 이상이라면 아래 행에서도 같은 값을 사용해야 함
                # 그래서 미래 행의 좌표를 spans에 미리 등록
                for future_row in range(row_index + 1, row_index + rowspan):
                    spans[(future_row, current_col)] = value
            # colspan만큼 열을 사용했으므로 현재 열 번호를 이동
            col_index += colspan

        # 행 마지막에 남은 rowspan 처리
        fill_rowspan()
        # 완성된 한 행을 전체 rows에 추가
        rows.append(row)

    # 모든 행의 길이를 동일하게 맞춤(가장 긴 행의 열 개수를 기준)
    max_columns = max((len(row) for row in rows), default=0)

    normalized_rows = []

    for row in rows:
        # 부족한 열은 빈 문자열로 채워 모든 행의 길이를 동일하게 만듦
        normalized_row = row + ([""] * (max_columns - len(row)))

        normalized_rows.append(normalized_row)
    # 정규화된 2차원 리스트 반환
    return normalized_rows

#### 본문의 표가 제대로 안뽑혀서 바이브 코딩 진행
하나의 table 아래쪽에 붙어 있는 부가 설명을 가져오는 역할

특히 `ul.char_list_box` 안의 `li` 내용을 찾고, 다음 table을 만나면 해당 표에 대한 설명 범위가 끝났다고 판단

In [ ]:
def get_table_description(table):
    """
    현재 table 아래에 존재하는 설명 문구를 추출하는 함수

    메이플스토리 페이지에서
    표 밑의 <ul class="char_list_box"> 영역을 찾아
    각각의 <li> 내용을 문자열로 반환한다.
    """
    # 여러 개의 설명 문장을 저장할 리스트
    descriptions = []

    # 현재 table 다음에 나오는 HTML 요소들을
    # DOM을 순서대로 하나씩 탐색
    for element in table.find_all_next():

        # 다음 table이 나타났다는 것은 현재 표의 설명 영역이 끝났다고 판단하고
        # 이후 설명은 다음 표의 내용일 수 있으므로 탐색 중단
        if element.name == "table":
            break

        # 현재 요소가
        # <ul class="char_list_box">
        # 형태인지 확인
        if (
            element.name == "ul"
            and "char_list_box" in element.get("class", [])
        ):
            # ul 바로 아래의 li만 가져옴(recursive=False를 사용해 중첩된 li까지 가져오는 것을 방지)
            for li in element.find_all("li", recursive=False):
                # li 내부 텍스트 추출
                text = li.get_text(" ", strip=True)
                # 내용이 있는 경우에만 저장
                if text:
                    descriptions.append(text)
    # 여러 개의 설명을 줄바꿈으로 연결하여 반환
    return "\n".join(descriptions)

### 상세 페이지를 최종 JSON 형태로 바꾸는 parser
페이지 설명, table, 제목, 표, 내용, 설명을 조합
- 최종 구조
```JSON
    {
        "name": 아이템/페이지 이름,
        "url": 페이지 URL,
        "description": 페이지 전체 설명,
        "tables": [
            {
                "table_index": 표 번호,
                "section_title": 표 제목,
                "rows": 표 데이터,
                "table_description": 표 아래 설명
            }
        ]
    }
```

In [ ]:
def parse_probability_info(html: str, url: str, name: str) -> Dict:
    """
    확률형 아이템 상세 페이지 HTML을 분석하여
    하나의 딕셔너리 형태로 반환한다.

    최종 구조:

    {
        "name": 아이템/페이지 이름,
        "url": 페이지 URL,
        "description": 페이지 전체 설명,
        "tables": [
            {
                "table_index": 표 번호,
                "section_title": 표 제목,
                "rows": 표 데이터,
                "table_description": 표 아래 설명
            }
        ]
    }
    """
    # HTML 문자열을 BeautifulSoup 객체로 변환
    soup = BeautifulSoup(html, "html.parser")
    # 실제 확률형 아이템 내용이 들어있는 본문 영역 선택
    content_area = soup.select_one("#container .contents_wrap")

    if content_area is None:
        print("본문 영역을 찾지 못했습니다.")
        return {}

    # -----------------------------
    # 페이지 전체 설명
    # -----------------------------
    # 설명 문장들을 임시 저장
    description_list = []
    # 본문에서 <ul> <li></li> </ul> 구조를 가져옴
    for li in content_area.select("ul.new_list > li"):
        # li의 텍스트 추출
        desc = li.get_text(" ", strip=True)

        if desc:
            description_list.append(desc)
    # 여러 설명 문장을 줄바꿈으로 합침
    description = "\n".join(description_list)

    # -----------------------------
    # table 추출
    # -----------------------------
    # 최종 table 정보가 저장될 리스트
    tables = []
    # 본문 안에 존재하는 모든 table 가져오기
    table_tags = content_area.find_all("table")
    # table마다 번호를 부여하면서 처리(start=1 : 첫 번째 table_index를 1부터 시작)
    for index, table in enumerate(table_tags, start=1):

        # -----------------------------
        # 테이블 제목
        # -----------------------------
        # 현재 table보다 앞쪽에서 가장 가까운 h2, h3, h4 태그를 찾음
        heading = table.find_previous(["h2", "h3", "h4"])

        if heading:

            # heading 내부의 직접적인 텍스트만 가져옴(recursive=False)
            section_title = "".join(
                heading.find_all(
                    string=True,
                    recursive=False
                )
            ).strip()

        else:
            section_title = ""

        # -----------------------------
        # rowspan / colspan 정규화
        # -----------------------------
        # rowspan/colspan을 펼쳐서 2차원 리스트 형태로 반환
        rows = normalize_html_table(table)

        # -----------------------------
        # 현재 table 아래 설명
        # -----------------------------

        table_description = get_table_description(table)

        # -----------------------------
        # table 저장
        # -----------------------------
        # 표에 실제 데이터가 있는 경우에만 저장
        if rows:

            tables.append({
                # 몇 번째 표인지
                "table_index": index,
                # 표의 제목
                "section_title": section_title,
                # 표 데이터
                "rows": rows,
                # 표 아래 설명
                "table_description": table_description,
            })

    # -----------------------------
    # 최종 결과
    # -----------------------------

    probability_info = {
        # get_probability_urls()에서 가져온 페이지 이름
        "name": name,
        # 해당 상세 페이지 URL
        "url": url,
        # 페이지 상단 설명
        "description": description,
        # 페이지 안의 모든 표
        "tables": tables,
    }

    # 하나의 상세 페이지에 대한 최종 데이터 반환
    return probability_info

#### 문서 수집

In [ ]:
# url 수집하는 함수 불러오기
# name과 url이 있는 리스트가 만들어짐
probability_items = get_probability_urls()
# 확률형 아이템 저장하는 리스트 선언
all_probability_items = []
# 상세페이지 하나씩 순회
for item in probability_items:
    # 현재 어떤 페이지를 수집 중인지 확인하기 위한 코드
    print("수집 중:", item["name"])
    # item["url"]의 상세 페이지에 GET 요청을 보내고 HTML 문자열을 가져옴
    html = get_probability_page(item["url"])
    # 요청 실패 시 get_probability_page()에서 빈 문자열 반환
    if not html:
        print("HTML 요청 실패")
        continue
    # item['url']을 가져와서 html 파싱하고 딕셔너리 형태로 반환
    probability_info = parse_probability_info(html, item["url"], item["name"])
    # 파싱에 성공하면 리스트에 저장
    if probability_info:
        all_probability_items.append(probability_info)

        print(
            f"{item['name']} 수집 완료 "
            f"- 테이블 "
            f"{len(probability_info['tables'])}개"
        )

print("\n전체 수집 페이지:", len(all_probability_items))

수집 중: 치장성
치장성 수집 완료 - 테이블 1개
수집 중: 로얄스타일
로얄스타일 수집 완료 - 테이블 1개
수집 중: 마스터피스 레드
마스터피스 레드 수집 완료 - 테이블 10개
수집 중: 마스터피스 블랙
마스터피스 블랙 수집 완료 - 테이블 10개
수집 중: 슈피겔만의 마법모자
슈피겔만의 마법모자 수집 완료 - 테이블 4개
수집 중: 미스틱 컬렉션
미스틱 컬렉션 수집 완료 - 테이블 1개
수집 중: 그랜드 컬렉션
그랜드 컬렉션 수집 완료 - 테이블 1개
수집 중: 장송의 프리렌 코디 컬렉션
장송의 프리렌 코디 컬렉션 수집 완료 - 테이블 1개
수집 중: 장송의 프리렌 굿즈 컬렉션
장송의 프리렌 굿즈 컬렉션 수집 완료 - 테이블 1개
수집 중: 부티크 기프트
부티크 기프트 수집 완료 - 테이블 2개
수집 중: 알쏭달쏭 라이딩 상자
알쏭달쏭 라이딩 상자 수집 완료 - 테이블 1개
수집 중: 알쏭달쏭 표정 얼굴장식 상자
알쏭달쏭 표정 얼굴장식 상자 수집 완료 - 테이블 1개
수집 중: 알쏭달쏭 코디 상자
알쏭달쏭 코디 상자 수집 완료 - 테이블 1개
수집 중: 뷰티
뷰티 수집 완료 - 테이블 1개
수집 중: 염색 일반 쿠폰
염색 일반 쿠폰 수집 완료 - 테이블 1개
수집 중: 알쏭달쏭 헤어 상자
알쏭달쏭 헤어 상자 수집 완료 - 테이블 1개
수집 중: 알쏭달쏭 성형 상자
알쏭달쏭 성형 상자 수집 완료 - 테이블 1개
수집 중: 게임
게임 수집 완료 - 테이블 1개
수집 중: 골드 애플
골드 애플 수집 완료 - 테이블 1개
수집 중: 플래티넘 애플
플래티넘 애플 수집 완료 - 테이블 1개
수집 중: 주문서/스크롤
주문서/스크롤 수집 완료 - 테이블 8개
수집 중: 추가 옵션
추가 옵션 수집 완료 - 테이블 4개
수집 중: 의문의 모몽
의문의 모몽 수집 완료 - 테이블 1개
수집 중: 상당한 탐험상자
상당한 탐험상자 수집 완료 - 테이블 3개
수집 중: 위대한 소울
위대한 소울 수집 완료 - 테이블 1개
수집 중: 소울 분해
소울 분해 수집 완료 - 테이블 2개


#### json 저장

In [27]:
def save_item_to_json(guides, BASE_PATH, file_name="maple_items.json"):

    file_path = os.path.join(BASE_PATH, file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(guides, f, ensure_ascii=False, indent=2)

    print("JSON 저장 완료")
    return file_path


save_item_to_json(all_probability_items, BASE_PATH)

JSON 저장 완료


'../../../data/RAG\\maple_items.json'